# Channel Exploration

Exploración del canal de YouTube del GCBA — Phase 1 de Avatar AI.

**Flujo:**
1. Fetch metadata del canal (sin descargar videos)
2. Análisis por playlist: duración, fps, resolución
3. Detección de subtítulos hardcodeados via OCR
4. Anotación manual / corrección
5. Resumen GO/NO-GO por playlist

## 1. Setup

In [ ]:
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, '../scripts')
from fetch_channel_catalog import load_config, build_catalog, save_catalog, load_catalog
from detect_hardcoded_subs import detect_hardcoded_subs

config = load_config('../config.yaml')
print('Config loaded')
print(f"  catalog path: {config['catalog']['path']}")
print(f"  channel url:  {config['channel']['url'] or '(not set)'}")

## 2. Fetch catálogo

Si ya existe un catálogo en `data/catalog/`, lo carga. Si no, corre el fetch.

> **Nota:** Configurá `channel.url` en `config.yaml` o sobreescribí `CHANNEL_URL` abajo.

In [ ]:
catalog = load_catalog(config)

if catalog:
    total = sum(p['n_videos'] for p in catalog['playlists'])
    print(f"Catálogo existente cargado ({catalog['fetched_at']})")
    print(f"  {len(catalog['playlists'])} playlists, {total} videos")
else:
    print('No se encontró catálogo. Ejecutá la celda siguiente para hacer el fetch.')

In [ ]:
# Ejecutar solo si no hay catálogo o querés refrescar.
# Sobreescribí CHANNEL_URL si no está en config.yaml
CHANNEL_URL = config['channel']['url']
# CHANNEL_URL = "https://www.youtube.com/@REEMPLAZAR"

if not CHANNEL_URL:
    raise ValueError("Configurá channel.url en config.yaml o sobreescribí CHANNEL_URL en esta celda")

catalog = build_catalog(CHANNEL_URL, config, existing=load_catalog(config))
save_catalog(catalog, config)
print(f"\nCatálogo generado: {len(catalog['playlists'])} playlists")

## 3. DataFrame y estadísticas por playlist

In [ ]:
rows = []
for pl in catalog['playlists']:
    for v in pl['videos']:
        rows.append({'playlist_title': pl['playlist_title'], 'playlist_id': pl['playlist_id'], **v})

df = pd.DataFrame(rows)
print(f"Total videos: {len(df)}")
df.head()

In [ ]:
def _modal(series):
    m = series.dropna().mode()
    return m.iloc[0] if len(m) else None

stats = df.groupby('playlist_title').agg(
    n_videos=('video_id', 'count'),
    total_min=('duration_sec', lambda x: round(x.sum() / 60, 1)),
    avg_min=('duration_sec', lambda x: round(x.mean() / 60, 1)),
    fps=('fps', _modal),
    resolution=('height', lambda x: f"{_modal(x)}p" if _modal(x) else 'n/a'),
    has_sub=('has_auto_subs', 'sum'),
).reset_index()

print('=== Stats por playlist ===')
stats

## 4. Videos sin metadata técnica (fps / resolución)

In [ ]:
missing = df[df['fps'].isna() | df['height'].isna()]
print(f"Videos sin fps/resolución en metadata: {len(missing)}")
if len(missing):
    missing[['playlist_title', 'video_id', 'title', 'duration_sec']].head(20)

## 5. Detección OCR de subtítulos hardcodeados

Para cada video sin `subtitle_type`, samplea frames y corre pytesseract en la franja inferior.

> **Requiere:** `tesseract-ocr` instalado en el sistema (`sudo apt install tesseract-ocr tesseract-ocr-spa`)  
> **Velocidad:** ~5-15 segundos por video según la conexión.

In [ ]:
pending = df[df['subtitle_type'].isna()]
print(f"Videos pendientes de clasificación: {len(pending)}")
pending[['playlist_title', 'video_id', 'title']].head(10)

In [ ]:
# Ejecutar para procesar todos los videos pendientes.
# Podés interrumpir y retomar — el catálogo se guarda al final.

# Limitá con .head(N) para probar con pocos videos primero:
to_process = df[df['subtitle_type'].isna()]
# to_process = to_process.head(5)  # descomentar para prueba

ocr_results = []
for i, (_, row) in enumerate(to_process.iterrows(), 1):
    print(f"[{i}/{len(to_process)}] {row['video_id']} — {row['title'][:50]}")
    try:
        r = detect_hardcoded_subs(row['url'], config)
        ocr_results.append({'video_id': row['video_id'], **r})
        print(f"  → {r['subtitle_type']:12s} | {r['frames_with_text']}/{r['frames_sampled']} frames | conf: {r['ocr_confidence_avg']}")
    except Exception as e:
        print(f"  ERROR: {e}")
        ocr_results.append({'video_id': row['video_id'], 'subtitle_type': 'error',
                            'ocr_confidence_avg': 0, 'sample_texts': []})

# Actualizar catálogo
ocr_by_id = {r['video_id']: r for r in ocr_results}
for pl in catalog['playlists']:
    for v in pl['videos']:
        if v['video_id'] in ocr_by_id:
            r = ocr_by_id[v['video_id']]
            v['subtitle_type'] = r['subtitle_type']
            v['ocr_confidence'] = r.get('ocr_confidence_avg')
            v['ocr_sample_text'] = ' | '.join(r.get('sample_texts', [])[:2])

save_catalog(catalog, config)
print('\nCatálogo actualizado con resultados OCR')

## 6. Anotación manual y corrección

Usá `MANUAL_OVERRIDES` para corregir clasificaciones incorrectas del OCR.  
Valores válidos: `"hardcoded"` | `"none"` | `"uncertain"`

In [ ]:
# Recargamos df con los resultados OCR
rows = []
for pl in catalog['playlists']:
    for v in pl['videos']:
        rows.append({'playlist_title': pl['playlist_title'], **v})
df = pd.DataFrame(rows)

# Videos que necesitan revisión manual
review = df[df['subtitle_type'].isin(['uncertain', 'error', None])]
print(f"Videos que necesitan revisión: {len(review)}")
review[['playlist_title', 'video_id', 'title', 'subtitle_type', 'ocr_confidence', 'ocr_sample_text']]

In [ ]:
# Completar con las correcciones manuales
MANUAL_OVERRIDES = {
    # "VIDEO_ID": "hardcoded",
    # "VIDEO_ID": "none",
}

if MANUAL_OVERRIDES:
    for pl in catalog['playlists']:
        for v in pl['videos']:
            if v['video_id'] in MANUAL_OVERRIDES:
                old = v['subtitle_type']
                v['subtitle_type'] = MANUAL_OVERRIDES[v['video_id']]
                print(f"  {v['video_id']}: {old} → {v['subtitle_type']}")
    save_catalog(catalog, config)
    print('Catálogo guardado con anotaciones manuales')
else:
    print('Sin overrides definidos')

## 7. Resumen GO/NO-GO por playlist

In [ ]:
rows = []
for pl in catalog['playlists']:
    for v in pl['videos']:
        rows.append({'playlist_title': pl['playlist_title'], **v})
df = pd.DataFrame(rows)

print('=== Distribución subtitle_type por playlist ===')
pivot = df.groupby(['playlist_title', 'subtitle_type']).size().unstack(fill_value=0)
print(pivot.to_string())

print('\n=== GO/NO-GO ===')
for pl_title in df['playlist_title'].unique():
    sub = df[df['playlist_title'] == pl_title]
    hardcoded = (sub['subtitle_type'] == 'hardcoded').sum()
    total = len(sub)
    status = 'GO   ' if hardcoded > 0 else 'NO-GO'
    pct = round(hardcoded / total * 100) if total else 0
    print(f"  [{status}] {pl_title[:50]:<50} {hardcoded}/{total} hardcoded ({pct}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subtitle type distribution
type_counts = df['subtitle_type'].fillna('sin_clasificar').value_counts()
axes[0].pie(type_counts, labels=type_counts.index, autopct='%1.0f%%', startangle=90)
axes[0].set_title('Distribución subtitle_type')

# Videos per playlist coloreados por hardcoded
pl_stats = df.groupby('playlist_title').apply(
    lambda g: pd.Series({
        'hardcoded': (g['subtitle_type'] == 'hardcoded').sum(),
        'none': (g['subtitle_type'] == 'none').sum(),
        'other': (~g['subtitle_type'].isin(['hardcoded', 'none'])).sum(),
    })
).head(15)

pl_labels = [t[:35] for t in pl_stats.index]
x = range(len(pl_labels))
axes[1].barh(pl_labels, pl_stats['hardcoded'], label='hardcoded', color='#2ecc71')
axes[1].barh(pl_labels, pl_stats['none'],
             left=pl_stats['hardcoded'], label='none', color='#e74c3c')
axes[1].barh(pl_labels, pl_stats['other'],
             left=pl_stats['hardcoded'] + pl_stats['none'], label='other', color='#95a5a6')
axes[1].set_title('Videos por playlist')
axes[1].set_xlabel('N videos')
axes[1].legend()

plt.suptitle('Canal GCBA — Exploración Phase 1', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f"\nCatálogo final guardado en: {config['catalog']['path']}/channel_catalog.json")
print("→ Input listo para Phase 2 (extracción OCR de videos con subtitle_type='hardcoded')")